In [3]:
import unsloth


In [29]:
from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer

# Charger le modèle quantifié 4bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
    device_map="auto",
    random_state = 42,
    use_cache = False,
)

# Assure-toi que les tokens sont bien définis
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id



device = model.device  # Le modèle est déjà sur le device approprié


==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.3.
   \\   /|    NVIDIA A100-PCIE-40GB MIG 1g.5gb. Num GPUs = 1. Max memory: 4.75 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [34]:
def generate_text_until_complete(prompt, max_new_tokens=400, max_iter=100):
    """
    Génère le texte de manière incrémentale jusqu'à ce qu'une phrase complète soit obtenue.
    Si le texte généré se termine par un fragment incomplet, on le retire dans le post-traitement.
    """
    # Tokenisation initiale
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Initialiser la séquence générée et le masque d'attention
    generated_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    
    for i in range(max_iter):
        # Générer UN token supplémentaire
        outputs = model.generate(
            input_ids=generated_ids,
            attention_mask=attention_mask,
            max_new_tokens=1,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            early_stopping=False,
        )
        
        # Extraire le nouveau token et le concaténer
        new_token = outputs[0][-1].unsqueeze(0).unsqueeze(0)
        generated_ids = torch.cat([generated_ids, new_token], dim=1)
        
        # Mettre à jour l'attention mask
        new_attention = torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device)
        attention_mask = torch.cat([attention_mask, new_attention], dim=1)
        
        # Décoder le texte généré jusqu'à présent
        text_generated = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        
        # Si le texte se termine par une ponctuation indiquant la fin d'une phrase, on peut arrêter
        if text_generated.rstrip().endswith(('.', '?', '!')):
            break
        
        # Pour éviter une boucle infinie, on arrête si on dépasse max_new_tokens
        if generated_ids.shape[1] >= max_new_tokens:
            break

    # Post-traitement : extraire uniquement les phrases complètes
    sentences = re.split(r'(?<=[.!?])\s+', text_generated.strip())
    if not text_generated.rstrip()[-1] in '.!?':
        # Retirer la dernière phrase incomplète
        sentences = sentences[:-1]
    final_text = " ".join(sentences)
    return final_text


In [35]:
# Exemple d’utilisation
prompt = "Les algorithmes sont"
generated = generate_text(prompt)
print("🧠 Texte généré :", generated)

Texte généré en détail :  Les algorithmes sont un outil essentiellement mathématique, en particulier les algorithme de recherche ont une application très large. Les algorithmiques sont utilisés dans la plupart des domaines comme l’analyse numérique et l’informatique. Il est par exemple très utile de faire des recherches sur le site internet pour trouver les meilleures offres. L’outil d’enregistrement qui vous permettra de savoir à quoi vous pouvez renommer votre entreprise et comment gérer vos affaires.
The Best Way to Research the Internet
In order for an online business to succeed, it needs to be as well organized and efficient as possible. If your company is not properly optimized then you will lose a lot of potential customers who are searching for what you offer but don’t know about it yet.
If there are many different kinds of SEO strategies in place, each one can have its own specific purpose or goal, which means that no matter
🧠 Texte généré : Les algorithmes sont un outil essen